In [21]:
import joblib
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

%run ../scripts/limpieza_datos.py

In [22]:
df = pd.read_csv('../datos/originales/credit_risk_dataset.csv')

train = limpieza_datos(df)

X = train.drop(columns=['estado_prestamo'])
y = train['estado_prestamo']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [23]:
pipeline = joblib.load('../modelos/Regresion_Logistica.joblib')

In [24]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1] 

print("--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

print("\n--- Matriz de Confusión ---")
print(confusion_matrix(y_test, y_pred))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metricas_cv = cross_val_score(
    estimator=pipeline, 
    X=X_train, 
    y=y_train, 
    cv=skf, 
    scoring='roc_auc',
    n_jobs=-1
)

# 4. Ver resultados
print(f"ROC AUC Folds: {metricas_cv}")
print(f"ROC AUC Score Promedio (Cross-Validation): {metricas_cv.mean():.4f}")

--- Reporte de Clasificación ---
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      4462
           1       0.74      0.49      0.59      1237

    accuracy                           0.85      5699
   macro avg       0.80      0.72      0.75      5699
weighted avg       0.84      0.85      0.84      5699


--- Matriz de Confusión ---
[[4245  217]
 [ 633  604]]
ROC AUC Folds: [0.85867272 0.86614534 0.853558   0.85217685 0.86550855]
ROC AUC Score Promedio (Cross-Validation): 0.8592
